# MCP Integration

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to connect LangChain agents to MCP servers using `langchain-mcp-adapters`, and build custom MCP servers with `FastMCP`.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph langchain-mcp-adapters

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Build a Custom MCP Server

Create a simple MCP server using `FastMCP` that exposes weather and calculator tools.

In [ ]:
%%writefile my_mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-tools")

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {
        "new york": "72°F, Sunny",
        "london": "58°F, Cloudy",
        "tokyo": "68°F, Clear",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

@mcp.tool()
def calculate_tip(bill_amount: float, tip_percentage: float) -> str:
    """Calculate the tip amount for a bill."""
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    return f"Tip: ${tip:.2f}, Total: ${total:.2f}"

if __name__ == "__main__":
    mcp.run(transport="stdio")

## 4. Connect to MCP Server with MultiServerMCPClient

Use `MultiServerMCPClient` to connect to the MCP server and retrieve tools.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4o-mini", model_provider="openai")

async with MultiServerMCPClient(
    {
        "my_tools": {
            "transport": "stdio",
            "command": "python",
            "args": ["my_mcp_server.py"],
        }
    }
) as client:
    tools = client.get_tools()
    print(f"Available tools: {[t.name for t in tools]}")

    agent = create_react_agent(
        model=model,
        tools=tools,
        prompt="You have access to weather and tip calculation tools.",
    )

    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]
    })
    print(result["messages"][-1].content)

## 5. Use the Calculator Tool

Ask the agent to calculate a tip using the MCP tool.

In [ ]:
async with MultiServerMCPClient(
    {
        "my_tools": {
            "transport": "stdio",
            "command": "python",
            "args": ["my_mcp_server.py"],
        }
    }
) as client:
    tools = client.get_tools()

    agent = create_react_agent(
        model=model,
        tools=tools,
        prompt="You have access to weather and tip calculation tools.",
    )

    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": "Calculate a 20% tip on a $85 bill"}]
    })
    print(result["messages"][-1].content)

## 6. Transport Types

Configure different transports for local and remote MCP servers.

In [ ]:
stdio_config = {
    "local_server": {
        "transport": "stdio",
        "command": "python",
        "args": ["my_mcp_server.py"],
    }
}

sse_config = {
    "remote_server": {
        "transport": "sse",
        "url": "http://localhost:8000/sse",
    }
}

http_config = {
    "http_server": {
        "transport": "streamable_http",
        "url": "http://localhost:8000/mcp",
    }
}

print("stdio: Local MCP servers run as child processes")
print("sse: Remote MCP servers over Server-Sent Events")
print("streamable_http: Remote MCP servers over HTTP streaming")

## 7. Multiple MCP Servers

Connect to several MCP servers simultaneously. Tools from all servers merge into one list.

In [ ]:
multi_config = {
    "weather_tools": {
        "transport": "stdio",
        "command": "python",
        "args": ["my_mcp_server.py"],
    },
}

async with MultiServerMCPClient(multi_config) as client:
    tools = client.get_tools()
    print(f"Total tools from all servers: {len(tools)}")
    for t in tools:
        print(f"  - {t.name}: {t.description}")

## Summary

- `langchain-mcp-adapters` bridges MCP servers with LangChain agents via `MultiServerMCPClient`
- Three transport types: `stdio`, `sse`, `streamable_http`
- `client.get_tools()` converts MCP tools into LangChain-compatible tools
- Build custom MCP servers with `FastMCP` and `@mcp.tool()`
- Connect to multiple MCP servers simultaneously for a unified tool set